# BRFSS 2024 Colorectal Cancer Screening: Data Preparation

**Project:** Predicting colorectal cancer screening non-compliance to guide targeted outreach  
**Analytic population:** BRFSS respondents aged 45-75 with a valid `_CRCREC3` value  
**Primary outcome:** Binary screening non-compliance  
**Secondary descriptive outcome:** Up to date vs overdue vs never screened



In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)


## 1. Paths and reproducibility settings


In [2]:
DATA_DIR = Path("../data")
RAW_DIR = DATA_DIR / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

RAW_URL = "https://drive.google.com/uc?id=1ktq0TuFxtKNtWLUBGnauNYJAgQTUmkv1"
RAW_ASCII_FILENAME = "LLCP2024.ASC"
RAW_ASCII_PATH = RAW_DIR / RAW_ASCII_FILENAME

DICT_PATH = DATA_DIR / "brfss2024_variable_dictionary.csv"
CODEBOOK_PATH = DATA_DIR / "USCODE24_LLCP_082125.HTML"
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RAW_CACHE_PATH = PROCESSED_DIR / "crc_selected_raw.csv"
ANALYTIC_PATH = PROCESSED_DIR / "crc_analytic_dataset.csv"
VARIABLE_DICTIONARY_PATH = PROCESSED_DIR / "crc_variable_dictionary.csv"
MISSINGNESS_PATH = PROCESSED_DIR / "crc_missingness_table.csv"
COHORT_FLOW_PATH = PROCESSED_DIR / "crc_cohort_flow.csv"
OUTCOME_RECODING_PATH = PROCESSED_DIR / "crc_outcome_recoding.csv"
SCREENING_STATUS_PATH = PROCESSED_DIR / "crc_screening_status_summary.csv"

EXPECTED_RAW_N = 457_670
EXPECTED_ELIGIBLE_N = 227_647
EXPECTED_COMPLIANT_N = 167_374
EXPECTED_OVERDUE_N = 16_343
EXPECTED_NEVER_N = 43_930

# Reproducibility: fix the seed used by any later sampling/imputation step.
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


def ensure_raw_file_available(raw_dir: Path, url: str) -> None:
    """Make sure a raw BRFSS ASCII file is available in `raw_dir`.

    Uses whatever file is already in `raw_dir` if present (e.g. when the
    repo's data/raw folder was populated manually). Otherwise downloads
    it from `url` (Google Drive) so the notebook also runs on a fresh
    checkout that only has the link.
    """
    existing = [
        path
        for path in raw_dir.iterdir()
        if path.is_file() and not path.name.startswith(".")
    ] if raw_dir.exists() else []
    if existing:
        return

    try:
        import gdown
    except ImportError as exc:
        raise ImportError(
            f"No local raw file found in {raw_dir} and gdown is not "
            "installed to download it. Install it with `pip install "
            "gdown`, or place the raw BRFSS ASCII file in the raw data "
            "folder yourself."
        ) from exc

    print(f"No local raw file found in {raw_dir}; downloading from {url} ...")
    gdown.download(url, str(raw_dir / RAW_ASCII_FILENAME), quiet=False)


ensure_raw_file_available(RAW_DIR, RAW_URL)

print("Data directory:", DATA_DIR)
print("Raw directory:", RAW_DIR)
print("Raw directory exists:", RAW_DIR.exists())
print("Variable dictionary exists:", DICT_PATH.exists())
print("Codebook exists:", CODEBOOK_PATH.exists())
print("Random seed:", RANDOM_SEED)

Data directory: ..\data
Raw directory: ..\data\raw
Raw directory exists: True
Variable dictionary exists: True
Codebook exists: True
Random seed: 42


## 2. Project variables and model domains


In [3]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

identifier_variables = [
    "_STATE",
    "SEQNO",
    "_AGE80",
    "RENTHOM1",
]

outcome_variables = [
    "_CRCREC3",
]

demographic_features = [
    "_AGEG5YR",
    "SEXVAR",
    "_RACEGR3",
    "_EDUCAG",
    "_INCOMG1",
    "EMPLOY1",
    "MARITAL",
    "_URBSTAT",
]

access_features = [
    "_HLTHPL2",
    "PERSDOC3",
    "MEDCOST1",
    "CHECKUP1",
]

health_features = [
    "GENHLTH",
    "DIABETE4",
    "_MICHD",
    "_SMOKER3",
    "_BMI5CAT",
]

model_features = (
    demographic_features
    + access_features
    + health_features
)

requested_variables = (
    identifier_variables
    + outcome_variables
    + model_features
)

assert len(model_features) == 17
assert len(requested_variables) == len(set(requested_variables))

print("Model predictors:", len(model_features))
print("Requested source fields:", len(requested_variables))


Model predictors: 17
Requested source fields: 22


## 3. Validate the BRFSS variable dictionary and locate the ASCII file


In [4]:
variable_dictionary = pd.read_csv(DICT_PATH)

required_dictionary_columns = {
    "variable", "label", "col_start", "col_end", "type"
}
assert required_dictionary_columns.issubset(variable_dictionary.columns)

available_variables = set(variable_dictionary["variable"])
missing_required = [
    variable
    for variable in requested_variables
    if variable not in available_variables
]
if missing_required:
    raise ValueError(
        f"Required variables missing from dictionary: {missing_required}"
    )

ascii_candidates = [
    path
    for path in RAW_DIR.iterdir()
    if path.is_file() and not path.name.startswith(".")
]
if not ascii_candidates:
    raise FileNotFoundError(f"No ASCII file found in {RAW_DIR}")

ASCII_PATH = max(
    ascii_candidates,
    key=lambda path: path.stat().st_size,
)

print("Selected ASCII file:", ASCII_PATH)
print(f"File size: {ASCII_PATH.stat().st_size / 1024**2:,.1f} MB")
display(
    variable_dictionary.loc[
        variable_dictionary["variable"].isin(requested_variables),
        ["variable", "label", "col_start", "col_end", "type"],
    ].sort_values("col_start")
)


Selected ASCII file: ..\data\raw\LLCP2024.ASC
File size: 900.4 MB


,variable,label,col_start,col_end,type
0,_STATE,State FIPS Code,1,2,num
7,SEQNO,Annual Sequence Number,36,45,str
28,SEXVAR,Sex of Respondent,88,88,num
29,GENHLTH,General Health,101,101,num
34,PERSDOC3,Have Personal Health Care Provider?,110,110,num
35,MEDCOST1,Could Not Afford To See Doctor,111,111,num
36,CHECKUP1,Length of time since last routine checkup,112,112,num
51,DIABETE4,(Ever told) you had diabetes,149,149,num
53,MARITAL,Marital Status,186,186,num
55,RENTHOM1,Own or Rent Home,188,188,num


## 4. Extract selected variables from the fixed-width BRFSS file

The first run creates `crc_selected_raw.csv`, containing the selected
source variables for all 457,670 respondents. Later runs use that cache
to avoid repeatedly scanning the 900 MB ASCII file. Delete the cache only
if the selected variables or the source file change.


In [5]:
selected_dictionary = (
    variable_dictionary.loc[
        variable_dictionary["variable"].isin(requested_variables)
    ]
    .copy()
    .sort_values("col_start")
    .reset_index(drop=True)
)

selected_dictionary["col_start"] = pd.to_numeric(
    selected_dictionary["col_start"]
)
selected_dictionary["col_end"] = pd.to_numeric(
    selected_dictionary["col_end"]
)

colspecs = [
    (int(start) - 1, int(end))
    for start, end in zip(
        selected_dictionary["col_start"],
        selected_dictionary["col_end"],
    )
]
fwf_names = selected_dictionary["variable"].tolist()

def standardise_source_types(frame):
    frame = frame.copy()
    frame["SEQNO"] = frame["SEQNO"].astype("string").str.strip()
    for column in requested_variables:
        if column == "SEQNO":
            continue
        frame[column] = pd.to_numeric(
            frame[column], errors="coerce"
        ).astype("Int64")
    return frame

if RAW_CACHE_PATH.exists():
    raw_selected = pd.read_csv(
        RAW_CACHE_PATH,
        dtype={"SEQNO": "string", "respondent_id": "string"},
        low_memory=False,
    )
    raw_selected = standardise_source_types(raw_selected)
    print("Loaded selected-variable cache:", RAW_CACHE_PATH)
else:
    raw_selected = pd.read_fwf(
        ASCII_PATH,
        colspecs=colspecs,
        names=fwf_names,
        header=None,
        dtype="string",
    )
    raw_selected = standardise_source_types(raw_selected)
    raw_selected.insert(
        0,
        "source_row_id",
        np.arange(1, len(raw_selected) + 1, dtype=np.int64),
    )
    raw_selected["respondent_id"] = (
        raw_selected["_STATE"].astype("string").str.zfill(2)
        + "-"
        + raw_selected["SEQNO"].str.zfill(10)
    )
    raw_selected.to_csv(RAW_CACHE_PATH, index=False)
    print("Created selected-variable cache:", RAW_CACHE_PATH)

if "source_row_id" not in raw_selected:
    raw_selected.insert(
        0,
        "source_row_id",
        np.arange(1, len(raw_selected) + 1, dtype=np.int64),
    )
if "respondent_id" not in raw_selected:
    raw_selected["respondent_id"] = (
        raw_selected["_STATE"].astype("string").str.zfill(2)
        + "-"
        + raw_selected["SEQNO"].str.zfill(10)
    )

print("Selected raw data shape:", raw_selected.shape)
display(raw_selected.head())


Created selected-variable cache: ..\data\processed\crc_selected_raw.csv
Selected raw data shape: (457670, 24)


,source_row_id,_STATE,SEQNO,SEXVAR,GENHLTH,PERSDOC3,MEDCOST1,CHECKUP1,DIABETE4,MARITAL,RENTHOM1,EMPLOY1,_URBSTAT,_HLTHPL2,_MICHD,_RACEGR3,_AGEG5YR,_AGE80,_BMI5CAT,_EDUCAG,_INCOMG1,_CRCREC3,_SMOKER3,respondent_id
0,1,1,2024000001,2,3,2,2,1,3,3,1,7,1,1,2,1,12,78,2,2,9,<NA>,4,01-2024000001
1,2,1,2024000002,1,1,1,2,1,3,1,1,7,1,1,1,1,13,80,3,4,7,<NA>,3,01-2024000002
2,3,1,2024000003,1,2,3,1,4,3,6,1,1,1,1,2,1,8,59,2,3,9,3,1,01-2024000003
3,4,1,2024000004,1,1,1,2,1,3,1,1,7,1,1,2,1,13,80,3,4,4,<NA>,4,01-2024000004
4,5,1,2024000005,1,3,1,2,1,3,5,1,8,1,1,2,1,6,47,2,3,2,3,4,01-2024000005


In [6]:
assert len(raw_selected) == EXPECTED_RAW_N
assert raw_selected["respondent_id"].notna().all()
assert raw_selected["respondent_id"].is_unique

raw_outcome_counts = (
    raw_selected["_CRCREC3"]
    .value_counts(dropna=False)
    .sort_index()
)
display(raw_outcome_counts.rename("N").to_frame())

assert raw_selected["_CRCREC3"].eq(1).sum() == EXPECTED_COMPLIANT_N
assert raw_selected["_CRCREC3"].eq(2).sum() == EXPECTED_OVERDUE_N
assert raw_selected["_CRCREC3"].eq(3).sum() == EXPECTED_NEVER_N
assert raw_selected["_CRCREC3"].isna().sum() == 230_023


,N
_CRCREC3,
1,167374
2,16343
3,43930
<NA>,230023


## 5. Define the eligible cohort and preserve all outcome states


In [7]:
eligible_mask = raw_selected["_CRCREC3"].isin([1, 2, 3])
analytic_raw = raw_selected.loc[eligible_mask].copy()

# _CRCREC3 already uses the CDC calculated eligibility definition.
# _AGE80 is retained as an independent cross-check.
age_check = analytic_raw["_AGE80"].between(45, 75, inclusive="both")

print("Eligible respondents:", len(analytic_raw))
print("Eligible records outside exact age 45-75:", (~age_check).sum())

assert len(analytic_raw) == EXPECTED_ELIGIBLE_N
assert age_check.all()


Eligible respondents: 227647
Eligible records outside exact age 45-75: 0


In [8]:
outcome_recode = pd.DataFrame(
    {
        "_CRCREC3": [1, 2, 3, pd.NA],
        "Original meaning": [
            "At least one recommended CRC test within interval",
            "Previously screened but not within interval",
            "Never had a recommended CRC test",
            "Missing or outside the eligible age definition",
        ],
        "Primary binary outcome": [
            "0 = Compliant",
            "1 = Non-compliant",
            "1 = Non-compliant",
            "Excluded",
        ],
        "Secondary descriptive status": [
            "Up to date",
            "Overdue",
            "Never screened",
            "Not in analytic cohort",
        ],
    }
)
display(outcome_recode)


,_CRCREC3,Original meaning,Primary binary outcome,Secondary descriptive status
0,1,At least one recommended CRC test within interval,0 = Compliant,Up to date
1,2,Previously screened but not within interval,1 = Non-compliant,Overdue
2,3,Never had a recommended CRC test,1 = Non-compliant,Never screened
3,<NA>,Missing or outside the eligible age definition,Excluded,Not in analytic cohort


In [9]:
binary_outcome_map = {
    1: 0,
    2: 1,
    3: 1,
}
detailed_outcome_map = {
    1: "Up to date",
    2: "Overdue",
    3: "Never screened",
}
noncompliance_type_map = {
    1: "Not applicable - compliant",
    2: "Overdue",
    3: "Never screened",
}

analytic_raw["crc_noncompliant"] = (
    analytic_raw["_CRCREC3"]
    .map(binary_outcome_map)
    .astype("Int64")
)
analytic_raw["crc_screening_status"] = (
    analytic_raw["_CRCREC3"]
    .map(detailed_outcome_map)
    .astype("string")
)
analytic_raw["noncompliance_type"] = (
    analytic_raw["_CRCREC3"]
    .map(noncompliance_type_map)
    .astype("string")
)

assert analytic_raw["crc_noncompliant"].isna().sum() == 0
assert analytic_raw["crc_noncompliant"].sum() == (
    EXPECTED_OVERDUE_N + EXPECTED_NEVER_N
)


In [10]:
screening_status_summary = (
    analytic_raw["crc_screening_status"]
    .value_counts()
    .rename_axis("Screening status")
    .reset_index(name="N")
)
screening_status_summary["Percent"] = (
    screening_status_summary["N"] / len(analytic_raw) * 100
).map(lambda x: f"{x:.2f}%")

noncompliant_only = analytic_raw.loc[
    analytic_raw["crc_noncompliant"].eq(1)
]
noncompliance_type_summary = (
    noncompliant_only["noncompliance_type"]
    .value_counts()
    .rename_axis("Non-compliance type")
    .reset_index(name="N")
)
noncompliance_type_summary["Percent among non-compliant"] = (
    noncompliance_type_summary["N"]
    / len(noncompliant_only)
    * 100
).map(lambda x: f"{x:.2f}%")

display(screening_status_summary)
display(noncompliance_type_summary)


,Screening status,N,Percent
0,Up to date,167374,73.52%
1,Never screened,43930,19.30%
2,Overdue,16343,7.18%


,Non-compliance type,N,Percent among non-compliant
0,Never screened,43930,72.89%
1,Overdue,16343,27.11%


## 6. Recode the 17 predictors and equity subgroup fields


In [12]:
value_maps = {
    "_AGEG5YR": {
        6: "45-49",
        7: "50-54",
        8: "55-59",
        9: "60-64",
        10: "65-69",
        11: "70-74",
        12: "75 (source category 75-79)",
    },
    "SEXVAR": {
        1: "Male",
        2: "Female",
    },
    "_RACEGR3": {
        1: "White only, non-Hispanic",
        2: "Black only, non-Hispanic",
        3: "Other race only, non-Hispanic",
        4: "Multiracial, non-Hispanic",
        5: "Hispanic",
    },
    "_EDUCAG": {
        1: "Did not graduate high school",
        2: "Graduated high school",
        3: "Attended college or technical school",
        4: "Graduated college or technical school",
    },
    "_INCOMG1": {
        1: "Less than $15,000",
        2: "$15,000 to < $25,000",
        3: "$25,000 to < $35,000",
        4: "$35,000 to < $50,000",
        5: "$50,000 to < $100,000",
        6: "$100,000 to < $200,000",
        7: "$200,000 or more",
    },
    "EMPLOY1": {
        1: "Employed for wages",
        2: "Self-employed",
        3: "Out of work for 1 year or more",
        4: "Out of work for less than 1 year",
        5: "Homemaker",
        6: "Student",
        7: "Retired",
        8: "Unable to work",
    },
    "MARITAL": {
        1: "Married",
        2: "Divorced",
        3: "Widowed",
        4: "Separated",
        5: "Never married",
        6: "Member of an unmarried couple",
    },
    "_URBSTAT": {
        1: "Urban",
        2: "Rural",
    },
    "_HLTHPL2": {
        1: "Insured",
        2: "Uninsured",
    },
    "PERSDOC3": {
        1: "Has personal doctor",
        2: "Has personal doctor",
        3: "No personal doctor",
    },
    "MEDCOST1": {
        1: "Yes",
        2: "No",
    },
    "CHECKUP1": {
        1: "Within past year",
        2: "1 to < 2 years",
        3: "2 to < 5 years",
        4: "5 or more years",
        8: "Never",
    },
    "GENHLTH": {
        1: "Excellent",
        2: "Very good",
        3: "Good",
        4: "Fair",
        5: "Poor",
    },
    "DIABETE4": {
        1: "Diabetes",
        2: "Diabetes during pregnancy only",
        3: "No diabetes",
        4: "Prediabetes or borderline diabetes",
    },
    "_MICHD": {
        1: "CHD or MI reported",
        2: "CHD or MI not reported",
    },
    "_SMOKER3": {
        1: "Current smoker - every day",
        2: "Current smoker - some days",
        3: "Former smoker",
        4: "Never smoked",
    },
    "_BMI5CAT": {
        1: "Underweight",
        2: "Normal weight",
        3: "Overweight",
        4: "Obese",
    },
    "RENTHOM1": {
        1: "Own",
        2: "Rent",
        3: "Other arrangement",
    },
}

clean_name_map = {
    "_AGEG5YR": "age_group",
    "SEXVAR": "sex",
    "_RACEGR3": "race_ethnicity",
    "_EDUCAG": "education_level",
    "_INCOMG1": "income_group",
    "EMPLOY1": "employment_status",
    "MARITAL": "marital_status",
    "_URBSTAT": "urban_rural",
    "_HLTHPL2": "insurance_status",
    "PERSDOC3": "personal_doctor",
    "MEDCOST1": "cost_barrier",
    "CHECKUP1": "checkup_recency",
    "GENHLTH": "general_health",
    "DIABETE4": "diabetes_status",
    "_MICHD": "heart_disease",
    "_SMOKER3": "smoking_status",
    "_BMI5CAT": "bmi_category",
    "RENTHOM1": "housing_tenure",
}

analytic_clean = analytic_raw.copy()

for source_variable, mapping in value_maps.items():
    clean_variable = clean_name_map[source_variable]
    analytic_clean[clean_variable] = (
        analytic_clean[source_variable]
        .map(mapping)
        .fillna("Not reported")
        .astype("string")
    )

analytic_clean["age_exact"] = analytic_clean["_AGE80"].astype("Int64")
analytic_clean["state_fips"] = (
    analytic_clean["_STATE"].astype("string").str.zfill(2)
)
analytic_clean["validation_group_state"] = analytic_clean["state_fips"]


display(analytic_clean.head())


,source_row_id,_STATE,SEQNO,SEXVAR,GENHLTH,PERSDOC3,MEDCOST1,CHECKUP1,DIABETE4,MARITAL,RENTHOM1,EMPLOY1,_URBSTAT,_HLTHPL2,_MICHD,_RACEGR3,_AGEG5YR,_AGE80,_BMI5CAT,_EDUCAG,_INCOMG1,_CRCREC3,_SMOKER3,respondent_id,crc_noncompliant,crc_screening_status,noncompliance_type,age_group,sex,race_ethnicity,education_level,income_group,employment_status,marital_status,urban_rural,insurance_status,personal_doctor,cost_barrier,checkup_recency,general_health,diabetes_status,heart_disease,smoking_status,bmi_category,housing_tenure,age_exact,state_fips,validation_group_state
2,3,1,2024000003,1,2,3,1,4,3,6,1,1,1,1,2,1,8,59,2,3,9,3,1,01-2024000003,1,Never screened,Never screened,55-59,Male,"White only, non-Hispanic",Attended college or technical school,Not reported,Employed for wages,Member of an unmarried couple,Urban,Insured,No personal doctor,Yes,5 or more years,Very good,No diabetes,CHD or MI not reported,Current smoker - every day,Normal weight,Own,59,01,01
4,5,1,2024000005,1,3,1,2,1,3,5,1,8,1,1,2,1,6,47,2,3,2,3,4,01-2024000005,1,Never screened,Never screened,45-49,Male,"White only, non-Hispanic",Attended college or technical school,"$15,000 to < $25,000",Unable to work,Never married,Urban,Insured,Has personal doctor,No,Within past year,Good,No diabetes,CHD or MI not reported,Never smoked,Normal weight,Own,47,01,01
5,6,1,2024000006,1,3,2,2,1,1,1,2,1,1,1,2,1,7,54,4,2,6,1,4,01-2024000006,0,Up to date,Not applicable - compliant,50-54,Male,"White only, non-Hispanic",Graduated high school,"$100,000 to < $200,000",Employed for wages,Married,Urban,Insured,Has personal doctor,No,Within past year,Good,Diabetes,CHD or MI not reported,Never smoked,Obese,Rent,54,01,01
6,7,1,2024000007,2,4,1,2,1,3,1,1,7,2,1,1,1,11,71,4,3,4,1,4,01-2024000007,0,Up to date,Not applicable - compliant,70-74,Female,"White only, non-Hispanic",Attended college or technical school,"$35,000 to < $50,000",Retired,Married,Rural,Insured,Has personal doctor,No,Within past year,Fair,No diabetes,CHD or MI reported,Never smoked,Obese,Own,71,01,01
7,8,1,2024000008,2,5,2,2,1,1,1,1,7,2,1,2,1,10,68,4,3,9,1,4,01-2024000008,0,Up to date,Not applicable - compliant,65-69,Female,"White only, non-Hispanic",Attended college or technical school,Not reported,Retired,Married,Rural,Insured,Has personal doctor,No,Within past year,Poor,Diabetes,CHD or MI not reported,Never smoked,Obese,Own,68,01,01


## 7. Missingness strategy

- `_CRCREC3` blanks are structural (ineligible age or incomplete screening
  inputs) and are excluded; the outcome is never imputed.
- Predictor refusals, don't-know responses and derived blanks are retained
  as **Not reported** in the primary analytic dataset.
- Income non-response is plausibly informative and is not imputed in the
  primary analysis.
- KNN or another imputation method may be tested only as a sensitivity
  analysis and must be fitted inside the training pipeline after the
  train/test split.


In [13]:
special_missing_codes = {
    "_AGEG5YR": {14},
    "SEXVAR": set(),
    "_RACEGR3": {9},
    "_EDUCAG": {9},
    "_INCOMG1": {9},
    "EMPLOY1": {9},
    "MARITAL": {9},
    "_URBSTAT": set(),
    "_HLTHPL2": {9},
    "PERSDOC3": {7, 9},
    "MEDCOST1": {7, 9},
    "CHECKUP1": {7, 9},
    "GENHLTH": {7, 9},
    "DIABETE4": {7, 9},
    "_MICHD": set(),
    "_SMOKER3": {9},
    "_BMI5CAT": set(),
    "RENTHOM1": {7, 9},
}

domain_lookup = {
    **{variable: "Demographic and socioeconomic"
       for variable in demographic_features},
    **{variable: "Healthcare access"
       for variable in access_features},
    **{variable: "Health status"
       for variable in health_features},
    "RENTHOM1": "Sensitivity analysis",
}

missingness_rows = [
    {
        "Variable": "_CRCREC3",
        "Clean variable": "crc_noncompliant",
        "Role": "Outcome",
        "Domain": "Outcome",
        "Analytic denominator": len(raw_selected),
        "Unusable N": int(raw_selected["_CRCREC3"].isna().sum()),
        "Unusable percent": (
            raw_selected["_CRCREC3"].isna().mean() * 100
        ),
        "Missingness type": (
            "Structural: outside eligibility or incomplete "
            "calculated screening status"
        ),
        "Primary decision": (
            "Exclude from analytic cohort; never impute"
        ),
    }
]

for variable in model_features + ["RENTHOM1"]:
    series = analytic_raw[variable]
    unusable = series.isna()
    codes = special_missing_codes[variable]
    if codes:
        unusable = unusable | series.isin(codes)

    missingness_rows.append(
        {
            "Variable": variable,
            "Clean variable": clean_name_map[variable],
            "Role": (
                "Predictor"
                if variable in model_features
                else "Auxiliary"
            ),
            "Domain": domain_lookup[variable],
            "Analytic denominator": len(analytic_raw),
            "Unusable N": int(unusable.sum()),
            "Unusable percent": unusable.mean() * 100,
            "Missingness type": (
                "Don't know/refused/derived blank"
            ),
            "Primary decision": (
                "Retain as Not reported; model-time "
                "imputation only as sensitivity analysis"
            ),
        }
    )

missingness_table = pd.DataFrame(missingness_rows)
missingness_table = missingness_table.sort_values(
    ["Role", "Unusable percent"],
    ascending=[True, False],
).reset_index(drop=True)

display(
    missingness_table.style.format(
        {
            "Analytic denominator": "{:,.0f}",
            "Unusable N": "{:,.0f}",
            "Unusable percent": "{:.2f}%",
        }
    )
)


,Variable,Clean variable,Role,Domain,Analytic denominator,Unusable N,Unusable percent,Missingness type,Primary decision
0,RENTHOM1,housing_tenure,Auxiliary,Sensitivity analysis,"227,647","1,535",0.67%,Don't know/refused/derived blank,Retain as Not reported; model-time imputation only as sensitivity analysis
1,_CRCREC3,crc_noncompliant,Outcome,Outcome,"457,670","230,023",50.26%,Structural: outside eligibility or incomplete calculated screening status,Exclude from analytic cohort; never impute
2,_INCOMG1,income_group,Predictor,Demographic and socioeconomic,"227,647","34,001",14.94%,Don't know/refused/derived blank,Retain as Not reported; model-time imputation only as sensitivity analysis
3,_BMI5CAT,bmi_category,Predictor,Health status,"227,647","13,268",5.83%,Don't know/refused/derived blank,Retain as Not reported; model-time imputation only as sensitivity analysis
4,_URBSTAT,urban_rural,Predictor,Demographic and socioeconomic,"227,647","7,745",3.40%,Don't know/refused/derived blank,Retain as Not reported; model-time imputation only as sensitivity analysis
5,_HLTHPL2,insurance_status,Predictor,Healthcare access,"227,647","6,329",2.78%,Don't know/refused/derived blank,Retain as Not reported; model-time imputation only as sensitivity analysis
6,_AGEG5YR,age_group,Predictor,Demographic and socioeconomic,"227,647","5,434",2.39%,Don't know/refused/derived blank,Retain as Not reported; model-time imputation only as sensitivity analysis
7,_RACEGR3,race_ethnicity,Predictor,Demographic and socioeconomic,"227,647","4,217",1.85%,Don't know/refused/derived blank,Retain as Not reported; model-time imputation only as sensitivity analysis
8,_SMOKER3,smoking_status,Predictor,Health status,"227,647","2,858",1.26%,Don't know/refused/derived blank,Retain as Not reported; model-time imputation only as sensitivity analysis
9,_MICHD,heart_disease,Predictor,Health status,"227,647","2,158",0.95%,Don't know/refused/derived blank,Retain as Not reported; model-time imputation only as sensitivity analysis


## 8. Cohort flow and project variable dictionary


In [14]:
cohort_flow = pd.DataFrame(
    {
        "Step": [
            1,
            2,
            3,
        ],
        "Cohort definition": [
            "All BRFSS 2024 respondents",
            (
                "Exclude blank _CRCREC3: outside age 45-75 "
                "or incomplete calculated screening status"
            ),
            "Final eligible analytic cohort",
        ],
        "Excluded N": [
            0,
            int(raw_selected["_CRCREC3"].isna().sum()),
            0,
        ],
        "Remaining N": [
            len(raw_selected),
            len(analytic_raw),
            len(analytic_raw),
        ],
    }
)
display(cohort_flow)


,Step,Cohort definition,Excluded N,Remaining N
0,1,All BRFSS 2024 respondents,0,457670
1,2,Exclude blank _CRCREC3: outside age 45-75 or i...,230023,227647
2,3,Final eligible analytic cohort,0,227647


In [15]:
feature_metadata = [
    ("_AGEG5YR", "age_group", "Demographic and socioeconomic",
     "Five-year age bands", "14 or blank", True),
    ("SEXVAR", "sex", "Demographic and socioeconomic",
     "Sex of respondent", "None expected", True),
    ("_RACEGR3", "race_ethnicity", "Demographic and socioeconomic",
     "Five-level race/ethnicity", "9", True),
    ("_EDUCAG", "education_level", "Demographic and socioeconomic",
     "Education category", "9", False),
    ("_INCOMG1", "income_group", "Demographic and socioeconomic",
     "Household income category", "9", True),
    ("EMPLOY1", "employment_status", "Demographic and socioeconomic",
     "Employment status", "9 or blank", False),
    ("MARITAL", "marital_status", "Demographic and socioeconomic",
     "Marital status", "9 or blank", False),
    ("_URBSTAT", "urban_rural", "Demographic and socioeconomic",
     "Urban/rural status", "Blank", False),
    ("_HLTHPL2", "insurance_status", "Healthcare access",
     "Any health insurance", "9", True),
    ("PERSDOC3", "personal_doctor", "Healthcare access",
     "Has a personal healthcare provider", "7, 9 or blank", False),
    ("MEDCOST1", "cost_barrier", "Healthcare access",
     "Could not afford doctor in past 12 months", "7, 9 or blank", False),
    ("CHECKUP1", "checkup_recency", "Healthcare access",
     "Time since routine check-up", "7 or 9", False),
    ("GENHLTH", "general_health", "Health status",
     "Self-rated general health", "7, 9 or blank", False),
    ("DIABETE4", "diabetes_status", "Health status",
     "Diabetes status", "7 or 9", False),
    ("_MICHD", "heart_disease", "Health status",
     "Ever reported CHD or MI", "Blank", False),
    ("_SMOKER3", "smoking_status", "Health status",
     "Calculated smoking status", "9", False),
    ("_BMI5CAT", "bmi_category", "Health status",
     "Calculated BMI category", "Blank", False),
]

project_variable_dictionary = pd.DataFrame(
    feature_metadata,
    columns=[
        "Source variable",
        "Clean variable",
        "Domain",
        "Concept",
        "Source missing codes",
        "Equity subgroup field",
    ],
)
project_variable_dictionary["Role"] = "Predictor"
project_variable_dictionary["Primary missing decision"] = (
    "Retain as Not reported"
)

outcome_dictionary_row = pd.DataFrame(
    [
        {
            "Source variable": "_CRCREC3",
            "Clean variable": "crc_noncompliant",
            "Domain": "Outcome",
            "Concept": "CRC screening non-compliance",
            "Source missing codes": "Blank / ineligible",
            "Equity subgroup field": False,
            "Role": "Outcome",
            "Primary missing decision": (
                "Exclude blank/ineligible; do not impute"
            ),
        }
    ]
)
project_variable_dictionary = pd.concat(
    [outcome_dictionary_row, project_variable_dictionary],
    ignore_index=True,
)

display(project_variable_dictionary)


,Source variable,Clean variable,Domain,Concept,Source missing codes,Equity subgroup field,Role,Primary missing decision
0,_CRCREC3,crc_noncompliant,Outcome,CRC screening non-compliance,Blank / ineligible,False,Outcome,Exclude blank/ineligible; do not impute
1,_AGEG5YR,age_group,Demographic and socioeconomic,Five-year age bands,14 or blank,True,Predictor,Retain as Not reported
2,SEXVAR,sex,Demographic and socioeconomic,Sex of respondent,None expected,True,Predictor,Retain as Not reported
3,_RACEGR3,race_ethnicity,Demographic and socioeconomic,Five-level race/ethnicity,9,True,Predictor,Retain as Not reported
4,_EDUCAG,education_level,Demographic and socioeconomic,Education category,9,False,Predictor,Retain as Not reported
5,_INCOMG1,income_group,Demographic and socioeconomic,Household income category,9,True,Predictor,Retain as Not reported
6,EMPLOY1,employment_status,Demographic and socioeconomic,Employment status,9 or blank,False,Predictor,Retain as Not reported
7,MARITAL,marital_status,Demographic and socioeconomic,Marital status,9 or blank,False,Predictor,Retain as Not reported
8,_URBSTAT,urban_rural,Demographic and socioeconomic,Urban/rural status,Blank,False,Predictor,Retain as Not reported
9,_HLTHPL2,insurance_status,Healthcare access,Any health insurance,9,True,Predictor,Retain as Not reported


## 9. Handoff for modelling, validation and targeted outreach

### Model-domain ablation

- **M1:** demographic and socioeconomic predictors
- **M2:** M1 plus healthcare-access predictors
- **M3:** M2 plus health-status predictors

### Validation

- Standard internal validation: stratified train/test split performed by
  the modelling team.
- Geographic internal-external validation: use
  `validation_group_state` with grouped or leave-state-out validation.
  `_STATE` must not be used as a predictor in the primary model.
- True external validation in Singapore is not possible with BRFSS alone;
  it requires a separate Singapore dataset and local recalibration.

### Calibration and outreach

Risk deciles must be created from out-of-sample predicted probabilities,
not during data preparation. The modelling team should report observed
non-compliance within each predicted-risk decile and define operational
tiers such as priority outreach (highest-risk decile) and secondary
outreach (next highest deciles).

### Equity subgroups

Use `insurance_status`, `income_group`, `race_ethnicity`, `sex` and
`age_group` to report subgroup discrimination, calibration and error
metrics. Retain `Not reported` as a visible group rather than silently
deleting it.


In [16]:
M1_CLEAN_FEATURES = [
    clean_name_map[variable]
    for variable in demographic_features
]
M2_CLEAN_FEATURES = M1_CLEAN_FEATURES + [
    clean_name_map[variable]
    for variable in access_features
]
M3_CLEAN_FEATURES = M2_CLEAN_FEATURES + [
    clean_name_map[variable]
    for variable in health_features
]

print("M1 features:", M1_CLEAN_FEATURES)
print("M2 features:", M2_CLEAN_FEATURES)
print("M3 features:", M3_CLEAN_FEATURES)

assert len(M1_CLEAN_FEATURES) == 8
assert len(M2_CLEAN_FEATURES) == 12
assert len(M3_CLEAN_FEATURES) == 17


M1 features: ['age_group', 'sex', 'race_ethnicity', 'education_level', 'income_group', 'employment_status', 'marital_status', 'urban_rural']
M2 features: ['age_group', 'sex', 'race_ethnicity', 'education_level', 'income_group', 'employment_status', 'marital_status', 'urban_rural', 'insurance_status', 'personal_doctor', 'cost_barrier', 'checkup_recency']
M3 features: ['age_group', 'sex', 'race_ethnicity', 'education_level', 'income_group', 'employment_status', 'marital_status', 'urban_rural', 'insurance_status', 'personal_doctor', 'cost_barrier', 'checkup_recency', 'general_health', 'diabetes_status', 'heart_disease', 'smoking_status', 'bmi_category']


## 10. Build the final analytic dataset and run quality checks


In [17]:
analytic_columns = [
    "respondent_id",
    "source_row_id",
    "state_fips",
    "validation_group_state",
    "age_exact",
    "_CRCREC3",
    "crc_noncompliant",
    "crc_screening_status",
    "noncompliance_type",
    *M3_CLEAN_FEATURES,
    "housing_tenure",
]

final_analytic = analytic_clean[analytic_columns].copy()

assert len(final_analytic) == EXPECTED_ELIGIBLE_N
assert final_analytic["respondent_id"].is_unique
assert final_analytic["crc_noncompliant"].isna().sum() == 0
assert set(final_analytic["crc_noncompliant"].unique()) == {0, 1}
assert final_analytic["age_exact"].between(45, 75).all()
assert final_analytic["crc_noncompliant"].sum() == 60_273
assert (
    final_analytic["crc_screening_status"].eq("Overdue").sum()
    == EXPECTED_OVERDUE_N
)
assert (
    final_analytic["crc_screening_status"].eq("Never screened").sum()
    == EXPECTED_NEVER_N
)
assert not set(M3_CLEAN_FEATURES).intersection(
    {"_CRCREC3", "respondent_id", "state_fips"}
)

# Every model predictor is complete after explicit Not reported recoding.
assert final_analytic[M3_CLEAN_FEATURES].isna().sum().sum() == 0

print("Final analytic shape:", final_analytic.shape)
print(
    "Non-compliance rate:",
    f"{final_analytic['crc_noncompliant'].mean():.2%}",
)
display(final_analytic.head())


Final analytic shape: (227647, 27)
Non-compliance rate: 26.48%


,respondent_id,source_row_id,state_fips,validation_group_state,age_exact,_CRCREC3,crc_noncompliant,crc_screening_status,noncompliance_type,age_group,sex,race_ethnicity,education_level,income_group,employment_status,marital_status,urban_rural,insurance_status,personal_doctor,cost_barrier,checkup_recency,general_health,diabetes_status,heart_disease,smoking_status,bmi_category,housing_tenure
2,01-2024000003,3,01,01,59,3,1,Never screened,Never screened,55-59,Male,"White only, non-Hispanic",Attended college or technical school,Not reported,Employed for wages,Member of an unmarried couple,Urban,Insured,No personal doctor,Yes,5 or more years,Very good,No diabetes,CHD or MI not reported,Current smoker - every day,Normal weight,Own
4,01-2024000005,5,01,01,47,3,1,Never screened,Never screened,45-49,Male,"White only, non-Hispanic",Attended college or technical school,"$15,000 to < $25,000",Unable to work,Never married,Urban,Insured,Has personal doctor,No,Within past year,Good,No diabetes,CHD or MI not reported,Never smoked,Normal weight,Own
5,01-2024000006,6,01,01,54,1,0,Up to date,Not applicable - compliant,50-54,Male,"White only, non-Hispanic",Graduated high school,"$100,000 to < $200,000",Employed for wages,Married,Urban,Insured,Has personal doctor,No,Within past year,Good,Diabetes,CHD or MI not reported,Never smoked,Obese,Rent
6,01-2024000007,7,01,01,71,1,0,Up to date,Not applicable - compliant,70-74,Female,"White only, non-Hispanic",Attended college or technical school,"$35,000 to < $50,000",Retired,Married,Rural,Insured,Has personal doctor,No,Within past year,Fair,No diabetes,CHD or MI reported,Never smoked,Obese,Own
7,01-2024000008,8,01,01,68,1,0,Up to date,Not applicable - compliant,65-69,Female,"White only, non-Hispanic",Attended college or technical school,Not reported,Retired,Married,Rural,Insured,Has personal doctor,No,Within past year,Poor,Diabetes,CHD or MI not reported,Never smoked,Obese,Own


## 11. Export reproducible data-preparation outputs


In [18]:
final_analytic.to_csv(ANALYTIC_PATH, index=False)
project_variable_dictionary.to_csv(
    VARIABLE_DICTIONARY_PATH, index=False
)
missingness_table.to_csv(MISSINGNESS_PATH, index=False)
cohort_flow.to_csv(COHORT_FLOW_PATH, index=False)
outcome_recode.to_csv(OUTCOME_RECODING_PATH, index=False)
screening_status_summary.to_csv(
    SCREENING_STATUS_PATH, index=False
)

exported_files = [
    ANALYTIC_PATH,
    VARIABLE_DICTIONARY_PATH,
    MISSINGNESS_PATH,
    COHORT_FLOW_PATH,
    OUTCOME_RECODING_PATH,
    SCREENING_STATUS_PATH,
]

for path in exported_files:
    print(
        path.name,
        f"{path.stat().st_size / 1024**2:,.2f} MB",
    )


crc_analytic_dataset.csv 67.98 MB
crc_variable_dictionary.csv 0.00 MB
crc_missingness_table.csv 0.00 MB
crc_cohort_flow.csv 0.00 MB
crc_outcome_recoding.csv 0.00 MB
crc_screening_status_summary.csv 0.00 MB


In [19]:
# Re-open the main output as a final integrity check.
check_df = pd.read_csv(
    ANALYTIC_PATH,
    dtype={
        "respondent_id": "string",
        "state_fips": "string",
        "validation_group_state": "string",
    },
    low_memory=False,
)

assert check_df.shape == final_analytic.shape
assert check_df["respondent_id"].is_unique
assert check_df["crc_noncompliant"].sum() == 60_273

print("Data preparation completed successfully.")
print("Main analytic dataset:", ANALYTIC_PATH)


Data preparation completed successfully.
Main analytic dataset: ..\data\processed\crc_analytic_dataset.csv
